# 第6回：モデルは本当に当たっているか

**今日の問い：手元のスコアをどこまで信じてよいか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

features = ["molecular_weight", "logp", "tpsa", "temperature_c", "reaction_time_h"]
clean = df.dropna(subset=features)
X_train, X_valid, y_train, y_valid = train_test_split(clean[features], clean["active"], test_size=0.25, random_state=42, stratify=clean["active"])


## TRY：木の深さと過学習


In [ ]:
rows = []
for depth in [1, 2, 4, 8, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    rows.append({
        "max_depth": str(depth),
        "学習スコア": accuracy_score(y_train, model.predict(X_train)),
        "検証スコア": accuracy_score(y_valid, model.predict(X_valid)),
    })
pd.DataFrame(rows).round(3)


## TRY：リークを入れると不自然に良くなる


In [ ]:
leak_features = [*features, "post_assay_signal"]
leaked = df.dropna(subset=leak_features)
X_train_l, X_valid_l, y_train_l, y_valid_l = train_test_split(leaked[leak_features], leaked["active"], test_size=0.25, random_state=42, stratify=leaked["active"])
leaked_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train_l, y_train_l)
print("リーク列ありの検証スコア:", round(accuracy_score(y_valid_l, leaked_model.predict(X_valid_l)), 3))
print("post_assay_signalは活性測定後の値なので、計画時の予測には使えません。")


## CHALLENGE：化合物系列を跨がせない分割


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, valid_idx = next(splitter.split(clean, groups=clean["scaffold_group"]))
print("学習側の系列:", sorted(clean.iloc[train_idx]["scaffold_group"].unique()))
print("検証側の系列:", sorted(clean.iloc[valid_idx]["scaffold_group"].unique()))


## リーク確認の3問

- その列は予測時点で存在するか
- 分割より前に全データから平均や変換を学習していないか
- 同じバッチ・日付・化合物系列が両側へ跨いでいないか
